In [21]:
import bs4
import os
from dotenv import load_dotenv
from operator import itemgetter

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [22]:
load_dotenv(dotenv_path='.env', override=True)

True

In [23]:
loader = WebBaseLoader(
    web_paths=("https://amanxai.com/2025/08/18/types-of-ai-agents-you-should-know/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("ct-container")
        )
    ),
)
blog_docs = loader.load()

In [24]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50
)
splits = text_splitter.split_documents(blog_docs)

In [25]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings()
)

In [26]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [27]:
template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [28]:
llm = ChatOpenAI(temperature=0, max_retries=3, request_timeout=60)

In [29]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [30]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [31]:
question = 'what is model-based agents'
response = rag_chain.invoke(question)

In [32]:
response

'Model-based agents are AI agents that keep track of what is happening in the world by maintaining an internal representation or model of the environment. This allows them to make better decisions, especially in situations where not everything is visible at once. They use their mental model to predict and fill in missing information, enabling them to make informed decisions in complex and uncertain environments.'